# Lab 2.3 — Building a Basic RAG System
**Module II · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-2-llm/lab2_3_basic_rag.ipynb)

---

## What you will do
Build a **Retrieval-Augmented Generation (RAG)** pipeline from scratch, piece by piece:

```
User question
     │
     ▼
 [Embed question]  →  Similarity search in vector store  →  Top-k chunks
                                                                   │
     ┌─────────────────────────────────────────────────────────────┘
     ▼
 [Build augmented prompt: context + question]  →  LLM  →  Grounded answer
```

By the end you will compare a plain LLM against your RAG system on company-specific questions — and see the difference clearly.

## Prerequisites
Labs 2.1 and 2.2 completed.

**Estimated time:** 70–90 min (core) + 20 min (extensions)

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
         "labs_assignment"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "labs_assignment/environment/requirements.txt"], check=True)
    sys.path.insert(0, "labs_assignment")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss

from utils.llm import SimpleLLM
from utils import load_company_kb

plt.rcParams["figure.dpi"] = 110
print("Imports OK.")

In [ ]:
llm = SimpleLLM()

---
## 1 · The knowledge base

We will work with the TechRetail policy documents introduced in Lab 2.1. Let's load and inspect them.

### Exercise 2.3.1 `[Core]` — Explore the knowledge base

Load the documents with `load_company_kb()` and print the title and first 100 characters of each document. How many documents are there?

In [ ]:
# YOUR CODE HERE
# Hint: Load the documents with load_company_kb() and print the title and first 100 characters of each document. How many doc...
raise NotImplementedError("Complete this exercise")

---
## 2 · Document chunking

**Why chunk?** In real applications, documents can be very long (entire PDFs, contracts, manuals). If we embed the whole document as a single vector, we lose fine-grained information — the query about shipping costs will match poorly against a document that also covers returns, loyalty points, and privacy.

Chunking splits each document into smaller, overlapping pieces so that each chunk is more focused and can be retrieved more precisely.

```
│←────── chunk 1 ──────→│
                  │←────── chunk 2 ──────→│
                                   │←────── chunk 3 ──────→│
         ←──────────── overlap ──────────→
```

### Exercise 2.3.2 `[Core]` — Implement a text chunker

Implement `chunk_text(text, max_chars, overlap_chars)` that:
1. Splits `text` into substrings of at most `max_chars` characters.
2. Each subsequent chunk starts `overlap_chars` characters before the end of the previous chunk (so context is not lost at boundaries).
3. Returns a list of strings.

Then write `chunk_documents(docs, max_chars=500, overlap_chars=100)` that applies `chunk_text` to each document and returns a flat list of chunk dicts (each with `title`, `source`, and `content` keys).

In [ ]:
# YOUR CODE HERE
# Hint: Implement chunk_text(text, max_chars, overlap_chars) that:

def chunk_text(text: str, max_chars: int = 500, overlap_chars: int = 100) -> list[str]:
    pass  # replace this

In [ ]:
chunks = chunk_documents(docs, max_chars=400, overlap_chars=80)
print(f"Documents: {len(docs)} → Chunks: {len(chunks)}")
print(f"Average chunk length: {np.mean([len(c['content']) for c in chunks]):.0f} chars")

# Show first two chunks from the first document
for c in chunks[:2]:
    print(f"\n[{c['id']}] {c['title']}")
    print(f"  {c['content'][:120]}…")

---
## 3 · Embeddings — turning text into vectors

An **embedding model** converts a piece of text into a fixed-size numerical vector. The key property: **semantically similar texts produce vectors that are close in space**.

We use `sentence-transformers/all-MiniLM-L6-v2` — a 22 MB model that produces 384-dimensional embeddings. It runs on CPU in milliseconds per sentence.

> **Reference:** The transformer-based embedding approach was popularised by Devlin et al. (BERT, NAACL 2019) and adapted for sentence-level similarity by Reimers & Gurevych (*Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*, EMNLP 2019).

In [ ]:
# Provided — no exercise here, just run it
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL} …")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

### Exercise 2.3.3 `[Core]` — Embed all chunks

Use `embedder.encode(texts, normalize_embeddings=True)` to embed all chunk contents. Store:
- `chunk_texts`: a list of strings (the `content` field of each chunk)
- `chunk_embeddings`: a 2-D NumPy array of shape `(n_chunks, 384)`

Normalized embeddings have unit length, which means dot product = cosine similarity.

In [ ]:
# YOUR CODE HERE
# Hint: Use embedder.encode(texts, normalize_embeddings=True) to embed all chunk contents. Store:
raise NotImplementedError("Complete this exercise")

---
## 4 · The vector store — building a FAISS index

**FAISS** (Facebook AI Similarity Search — Johnson et al., IEEE 2021) is a library for fast nearest-neighbour search over dense vectors. We use `IndexFlatIP` (flat inner product index):
- "Flat" = brute-force search — exact results, no approximation. Fine for small KBs.
- "IP" = inner product. With normalised embeddings, inner product equals cosine similarity.

For millions of vectors you would use approximate indexes (IVF, HNSW), but `IndexFlatIP` is perfect for our 8 documents.

### Exercise 2.3.4 `[Core]` — Build the FAISS index and implement `search`

1. Create a `faiss.IndexFlatIP(dim)` where `dim` is the embedding dimension.
2. Add all chunk embeddings to the index with `index.add()`.
3. Implement `search(query, top_k=3)` that:
   - Embeds the `query` string (normalised).
   - Calls `index.search(query_embedding, top_k)` to get distances and indices.
   - Returns a list of dicts: each has `content`, `title`, `source`, and `score`.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Create a faiss.IndexFlatIP(dim) where dim is the embedding dimension.
raise NotImplementedError("Complete this exercise")

In [ ]:
# Test the search
test_query = "How do I return a laptop?"
results = search(test_query, top_k=3)

print(f"Query: '{test_query}'\n")
for r in results:
    print(f"  Score: {r['score']:.3f} | [{r['id']}] {r['title']}")
    print(f"  {r['content'][:120]}…\n")

> **Good sign:** The top result should be the Return Policy document with a high cosine similarity score. Notice that the search works by *semantic meaning* — not just keyword matching. Try 'Can I get my money back?' and observe it still finds the returns policy.

---
## 5 · The full RAG pipeline

### Exercise 2.3.5 `[Core]` — Build the augmented prompt

Implement `build_prompt(query, retrieved_chunks)` that formats the retrieved chunks and the question into a single prompt string:

```
Answer the question below using ONLY the context provided.
If the answer is not in the context, say "I don't have that information."

Context:
[1] Return and Refund Policy (Customer Policy Manual v2.4 — Section 4):
... chunk content ...

[2] Shipping and Delivery (Customer Policy Manual v2.4 — Section 5):
... chunk content ...

Question: <query>
Answer:
```

In [ ]:
# YOUR CODE HERE
# Hint: Implement build_prompt(query, retrieved_chunks) that formats the retrieved chunks and the question into a single prom...

def build_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    pass  # replace this

### Exercise 2.3.6 `[Core]` — Assemble the full RAG function

Implement `rag_answer(query, top_k=3)` that chains everything together:
1. `search(query, top_k)` to retrieve relevant chunks.
2. `build_prompt(query, results)` to build the augmented prompt.
3. `llm.generate(prompt, temperature=0)` to generate the answer.
4. Return the answer string.

In [ ]:
# YOUR CODE HERE
# Hint: Implement rag_answer(query, top_k=3) that chains everything together:

def rag_answer(query: str, top_k: int = 3) -> str:
    pass  # replace this

---
## 6 · The big comparison: LLM alone vs. RAG

This is the centrepiece of the lab. We run the same three questions through two pipelines and compare the answers against the known ground truth.

### Exercise 2.3.7 `[Core]` — Side-by-side comparison

For each question in the table below, get the answer from:
- `llm.generate(question, temperature=0)` → plain LLM
- `rag_answer(question)` → RAG pipeline

Print both and mark whether each is correct.

In [ ]:
# YOUR CODE HERE
# Hint: For each question in the table below, get the answer from:
raise NotImplementedError("Complete this exercise")

> **What you should see:**
> - The plain LLM gives a generic or plausible-but-wrong answer (wrong numbers, hedged statements like "typically 30 days", or fabricated prices in a currency it guesses).
> - The RAG answer is accurate because it retrieved the correct policy excerpt and grounded the response in it.
>
> This is the core of the course's argument: **the LLM's language capability plus reliable retrieval outperforms the LLM's memory alone** for knowledge-intensive tasks over private or domain-specific data.

---
## 7 · `[Extension]` Visualising the embedding space

Embeddings live in 384 dimensions — impossible to visualise directly. We can project them down to 2D with **PCA** to see whether semantically related chunks cluster together.

### Exercise 2.3.8 `[Extension]` — PCA projection of chunk embeddings

Use `sklearn.decomposition.PCA` to project all chunk embeddings to 2D. Plot each point labelled by document title. Then embed a few query strings and plot them as stars, to see that they land near their relevant chunks.

In [ ]:
# YOUR CODE HERE
# Hint: Use sklearn.decomposition.PCA to project all chunk embeddings to 2D. Plot each point labelled by document title. Then...
raise NotImplementedError("Complete this exercise")

> **What to look for:** Chunks from the same document should cluster together (similar content = similar vectors). Query points should land near the document chunks they are semantically related to. This visualisation confirms that similarity search in the full 384-D space is doing something geometrically meaningful.

---
## 8 · `[Extension]` The limit of RAG: multi-hop questions

This section is a deliberate cliffhanger — designed to motivate Module III and IV.

### Exercise 2.3.9 `[Extension]` — Ask a question that requires connecting two documents

Try the question below with `rag_answer()`. Note how many reasoning *hops* are needed to answer it correctly, and whether RAG succeeds.

In [ ]:
# YOUR CODE HERE
# Hint: Try the question below with rag_answer(). Note how many reasoning hops are needed to answer it correctly, and whether...
raise NotImplementedError("Complete this exercise")

> **Why RAG struggles here:** This question requires:
> 1. Retrieving the Premium discount (10%) from one document.
> 2. Retrieving the loyalty points rate (1.5× for Premium members) from another.
> 3. **Composing** both facts with arithmetic.
>
> RAG retrieves individual chunks well, but connecting information *across* documents through multi-step reasoning is unreliable. The LLM may get one fact right and fabricate the other, or make arithmetic errors.
>
> In the real world, an even harder version of this problem involves entities connected by **relationships** in a database — not just two text documents, but a graph of interconnected records. That is precisely what **Module III (GNNs)** and **Module IV (GraphRAG)** address: reasoning over structured, relational data in a way that plain RAG cannot.

---
## Summary

| RAG component | What we built | Reference |
|---|---|---|
| **Chunking** | Overlapping fixed-size text windows | — |
| **Embedding** | `all-MiniLM-L6-v2` → 384-D normalised vectors | Reimers & Gurevych, EMNLP 2019 |
| **Vector store** | `faiss.IndexFlatIP` (exact cosine similarity) | Johnson et al., IEEE 2021 |
| **Retrieval** | Top-k nearest neighbours in embedding space | — |
| **Augmentation** | Context injection into the prompt | Lewis et al., NeurIPS 2020 |
| **Generation** | `SimpleLLM` with grounded context | — |

**What RAG fixes:** factual accuracy on domain-specific questions.

**What RAG does not fix:** multi-hop reasoning across related entities — which requires graphs.

**Next → Module III:** Graph Neural Networks — the data structure and learning algorithm that captures relationships, and how to learn from them.